# Parser Debug Notebook

Use this notebook to interactively test and debug the PDF parsers.

**Purpose:**
- Inspect raw PDF text extracted by pdfplumber
- Test RBC Visa and BMO Mastercard parsers on individual files
- Verify transaction line detection patterns
- Debug date parsing, amount parsing, and foreign currency handling

In [ ]:
import sys
from pathlib import Path

# Add expense_elt to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print('Project root:', project_root)

In [ ]:
from ingestion.pdf_reader import read_pdf
from ingestion.rbc_parser import parse_rbc_pdf
from ingestion.bmo_parser import parse_bmo_pdf

# List available PDFs
rbc_dir = project_root / 'data' / 'RBC_Visa'
bmo_dir = project_root / 'data' / 'BMO_Mastercard'

rbc_pdfs = sorted(rbc_dir.glob('*.pdf'))
bmo_pdfs = sorted(bmo_dir.glob('*.pdf'))

print(f'RBC PDFs ({len(rbc_pdfs)}):')
for p in rbc_pdfs:
    print(f'  {p.name}')

print(f'\nBMO PDFs ({len(bmo_pdfs)}):')
for p in bmo_pdfs:
    print(f'  {p.name}')

In [ ]:
# --- Inspect raw PDF text ---
# Change this to the PDF you want to debug
pdf_path = rbc_pdfs[0] if rbc_pdfs else None

if pdf_path:
    pages = read_pdf(pdf_path)
    print(f'Pages in {pdf_path.name}: {len(pages)}')
    print('\n--- Page 1 text ---')
    print(pages[0][1][:3000])  # first 3000 chars

In [ ]:
# --- Parse a single RBC PDF and inspect results ---
if rbc_pdfs:
    txns = parse_rbc_pdf(rbc_pdfs[0])
    print(f'Parsed {len(txns)} transactions from {rbc_pdfs[0].name}')
    print()
    for t in txns[:10]:
        print(f"  {t['transaction_date_raw']}  {t['merchant_raw'][:50]:<50}  {t['amount_raw']}")
    if len(txns) > 10:
        print(f'  ... and {len(txns) - 10} more')

In [ ]:
# --- Parse a single BMO PDF and inspect results ---
if bmo_pdfs:
    txns = parse_bmo_pdf(bmo_pdfs[0])
    print(f'Parsed {len(txns)} transactions from {bmo_pdfs[0].name}')
    print()
    for t in txns[:10]:
        print(f"  {t['transaction_date_raw']}  {t['merchant_raw'][:50]:<50}  {t['amount_raw']}  [{t.get('cardholder', '')}]")
    if len(txns) > 10:
        print(f'  ... and {len(txns) - 10} more')

In [ ]:
# --- Debug specific lines using regex directly ---
import re

# Test RBC transaction line regex
test_lines = [
    'DEC 09  DEC 10  AMAZON.CA*ZR3WI9700 AMAZON.CA ON     $23.00',
    'JAN 01  JAN 02  NETFLIX.COM/BILL 866-716-0414 CA    $20.99',
    'DEC 31  JAN 01  STARBUCKS #12345 VANCOUVER BC    $6.75',
]

from ingestion.rbc_parser import TRANS_LINE_RE
for line in test_lines:
    m = TRANS_LINE_RE.match(line)
    if m:
        print(f'MATCH: {m.groups()}')
    else:
        print(f'NO MATCH: {line}')